# Mestrado em Inteligência Artificial 25/26

# K-Armed Bandit Demo

This notebook demonstrates the implementation of the K-Armed Bandit environment and various agent strategies described in Sutton & Barto's Reinforcement Learning book.

Specifically, we compare:
1. **$\epsilon$-Greedy Agents** with varying degrees of exploration ($\epsilon = 0$, $\epsilon = 0.01$, and $\epsilon = 0.1$).
2. **Optimistic Initial Value Agents** vs. **Upper Confidence Bound (UCB)** action selection.
3. **Gradient Bandit Agents** with and without baseline.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import matplotlib.pyplot as plt
from mia_rl.envs.kbandits import KArmedBandit
from mia_rl.agents.kbandits import EpsilonGreedy, UCB, GradientBandit

### Experiment Runner Helper Function

In [ ]:
def run_experiment(agent, env, steps=1000, runs=2000):
    rewards = np.zeros((runs, steps))
    optimal = np.zeros((runs, steps))

    for r in range(runs):
        env.reset()
        agent.reset()

        for t in range(steps):
            action = agent.select_action()
            _, reward, _ = env.step(action)
            agent.update(action, reward)

            rewards[r, t] = reward
            optimal[r, t] = (action == env.optimal_action)

    return rewards.mean(axis=0), optimal.mean(axis=0)

### 1. $\epsilon$-Greedy Comparison
We evaluate how the agent behaves with different exploration probabilities: $\epsilon \in \{0, 0.01, 0.1\}$.

In [ ]:
steps, runs = 1000, 500  # 500 runs for fast notebook execution
env = KArmedBandit(k=10)
epsilons = [0, 0.01, 0.1]

plt.figure(figsize=(10, 5))
for eps in epsilons:
    print(f"Running epsilon={eps}...")
    agent = EpsilonGreedy(epsilon=eps)
    rewards, _ = run_experiment(agent, env, steps, runs)
    plt.plot(rewards, label=f"$\epsilon$ = {eps}")

plt.xlabel("Steps")
plt.ylabel("Average reward")
plt.title("$\epsilon$-greedy Comparison on 10-armed Bandit")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 2. Optimistic Initialization vs. UCB (Upper Confidence Bound)
- **Optimistic Greedy**: Sets initial action values $Q_0(a) = 5$ to encourage early exploration.
- **UCB**: Uses upper confidence bounds on action value estimates to guide exploration under uncertainty.

In [ ]:
agents = {
    "Optimistic greedy ($Q_0=5, \epsilon=0$)": EpsilonGreedy(epsilon=0, optimistic=5),
    "UCB ($c=2$)": UCB(c=2),
    "Realistic greedy ($Q_0=0, \epsilon=0.1$)": EpsilonGreedy(epsilon=0.1, optimistic=0),
}

plt.figure(figsize=(10, 5))
for name, agent in agents.items():
    print(f"Running {name}...")
    rewards, _ = run_experiment(agent, env, steps, runs)
    plt.plot(rewards, label=name)

plt.xlabel("Steps")
plt.ylabel("Average reward")
plt.title("Optimistic vs UCB vs Epsilon-Greedy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 3. Gradient Bandit Agents
Gradient bandits learn action preferences rather than action values. Using a baseline (the average reward obtained so far) stabilizes learning.

In [ ]:
gradient_agents = {
    "$\alpha=0.1$ with baseline": GradientBandit(alpha=0.1, baseline=True),
    "$\alpha=0.4$ with baseline": GradientBandit(alpha=0.4, baseline=True),
    "$\alpha=0.1$ no baseline": GradientBandit(alpha=0.1, baseline=False),
}

plt.figure(figsize=(10, 5))
for name, agent in gradient_agents.items():
    print(f"Running {name}...")
    rewards, _ = run_experiment(agent, env, steps, runs)
    plt.plot(rewards, label=name)

plt.xlabel("Steps")
plt.ylabel("Average reward")
plt.title("Gradient Bandit Methods")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()